[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 06](README.md)

# CUDA: bibliotecas, streams y perfiles

**Tema:** 06 · **Sesiones:** 28, 29, 30 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cuándo usar una biblioteca acelerada y cómo separar preparación, transferencia y ejecución repetida?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** Una biblioteca acelerada aporta algoritmos afinados, pero su ventaja depende de layout, tamaños, workspace, reutilización y residencia de datos.

**Prerrequisitos.**

- C++20, memoria y descomposición por datos.
- Modelo host–dispositivo y medición extremo a extremo.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Seleccionar Thrust/CUB o una biblioteca de dominio.
- Administrar handle, plan, descriptor, workspace y stream.
- Interpretar Nsight y tiempo extremo a extremo sin ocultar preparación.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Las bibliotecas aprovechan algoritmos y kernels especializados cuando layout, tipo y tamaño coinciden con su contrato.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Planes y workspaces se reutilizan; crearlos dentro de cada repetición distorsiona la medición.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Una comparación justa mantiene operación matemática, precisión, tolerancia, datos residentes y costos incluidos.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- SIMT — ejecución de hilos agrupados sobre una instrucción
- warp — grupo de hilos planificado conjuntamente
- coalescencia — agrupación eficiente de accesos contiguos
- tile — bloque de datos reutilizado localmente


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Offload Host Device

![Flujo de datos entre host y dispositivo](../../images/offload-host-device.svg)

**Cómo leerlo.** Separa preparación, H2D, kernel, D2H y validación. Esa separación evita llamar tiempo total a una medición que solo cubre el kernel.

### Metodo Rendimiento

![Ciclo de medición, resumen, perfil e hipótesis](../../images/metodo-rendimiento.svg)

**Cómo leerlo.** Una medición se repite y resume antes de perfilar. La conclusión genera un experimento nuevo cambiando una sola variable controlada.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "06"
NOTEBOOK = "06_cuda/03_bibliotecas_perfiles.ipynb"
assert (ROOT / "curso" / "notebooks" / "06_cuda" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Selección inicial

**Situación.** Una función explícita evita recomendar una biblioteca sin considerar la operación.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
def candidate(operation):
    return {
        "transform": "Thrust",
        "reduce": "CUB/Thrust",
        "gemm": "cuBLAS/cuBLASLt",
        "fft": "cuFFT",
        "spmv": "cuSPARSE",
        "dense_solve": "cuSOLVER",
        "random": "cuRAND",
    }.get(operation, "kernel propio o composición")
operations = ("transform", "reduce", "gemm", "fft", "spmv", "dense_solve", "random", "stencil_fused")
assert candidate("gemm") == "cuBLAS/cuBLASLt"
assert candidate("stencil_fused") == "kernel propio o composición"
for operation in operations: print(f"{operation:14} -> {candidate(operation)}")


### Explicación del resultado

La selección final incorpora tamaños, layout, residencia, precisión, repetición y posibilidad de fusión.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Perfil de fases

**Situación.** Se resume una serie repetida mediante mediana y se separa preparación.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
import statistics
phases = {
    "plan_once": [3.4],
    "h2d": [1.1, 1.0, 1.2, 1.1, 1.0],
    "compute": [0.42, 0.40, 0.41, 0.43, 0.40],
    "d2h": [0.8, 0.82, 0.79, 0.81, 0.8],
}
medians = {name: statistics.median(values) for name, values in phases.items()}
repeated_total = medians["h2d"] + medians["compute"] + medians["d2h"]
print(medians, "total_repetido_ms", repeated_total)
assert repeated_total > medians["compute"]


### Lectura razonada

Se publican tanto ejecución con datos residentes como extremo a extremo; el plan se amortiza solo cuando se reutiliza.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué costos deben incluirse simétricamente al comparar un kernel propio con una biblioteca?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Implementar el ciclo crear–configurar–ejecutar–validar–destruir.
2. Comparar una biblioteca con referencia y kernel correcto.
3. Conservar comandos Nsight y métricas que respondan la pregunta.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Incluir creación de plan solo en una variante.
- Comparar column-major con row-major sin ajustar layout.
- Usar ocupación alta como sinónimo de rendimiento.


## Criterios de aceptación

- Estados de biblioteca comprobados.
- Recursos liberados y streams documentados.
- Preparación, ejecución residente y total separados.


## Síntesis

- La pregunta que debes poder responder es: **¿Cuándo usar una biblioteca acelerada y cómo separar preparación, transferencia y ejecución repetida?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Guía y bibliotecas CUDA](README.md)
- [Ejemplos CUDA](../../ejemplos/06_cuda/README.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 06](README.md)
